In [1]:
import altair as alt
import numpy as np
import pandas as pd

/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Scatterplots

In [25]:
df = pd.read_excel('routes_03012026.xlsx')

In [39]:
route_colors = df.set_index('route')['_color'].to_dict()

chart_1 = alt.Chart(df).mark_circle(size=60).encode(
    x='miles_approx', y=alt.Y('time', title='Time (hrs)'), 
    color=alt.Color('route:N', scale=alt.Scale(domain=list(route_colors.keys()),
                                             range=list(route_colors.values()))),
    tooltip=['route_label', 'time', 'frequency', 'miles_approx', 'n_stations_approx']
    ).properties(width=250, height=250)

chart_2 = alt.Chart(df).mark_circle(size=60).encode(
    x='miles_approx', y='n_stations_approx', 
    color=alt.Color('route:N', scale=alt.Scale(domain=list(route_colors.keys()),
                                             range=list(route_colors.values()))),
    tooltip=['route_label', 'time', 'frequency', 'miles_approx', 'n_stations_approx']
    ).properties(width=250, height=250)

chart_3 = alt.Chart(df).mark_circle(size=60).encode(
    x='n_stations_approx', y=alt.Y('time', title='Time (hrs)'), 
    color=alt.Color('route:N', scale=alt.Scale(domain=list(route_colors.keys()),
                                             range=list(route_colors.values()))),
    tooltip=['route_label', 'time', 'frequency', 'miles_approx', 'n_stations_approx']
    ).properties(width=250, height=250)


alt.hconcat(chart_1, chart_2, chart_3)

alt.HConcatChart(...)

In [80]:
# Route color dictionary
route_colors = df.set_index('route')['_color'].to_dict()

# Shared selection point
highlight = alt.selection_point(name='route', fields=['route', 'route_label'], on='click', empty='none')

title = alt.TitleParams(
    text={
        "expr": "route.route ? 'Route: '+route.route_label : 'Select a route'"
    }
)

circle_size = 80
chart_width=250
chart_height=250
# -----------------------------
# Chart 1: Time vs Miles
# -----------------------------
base1 = alt.Chart(df).mark_circle(size=circle_size).encode(
    x=alt.X('miles_approx', title='Distance (mi)'),
    y=alt.Y('time', title='Duration (hrs)'),
    color=alt.Color('route:N', legend=None, scale=alt.Scale(
        domain=list(route_colors.keys()),
        range=list(route_colors.values())
    )),
    tooltip=['route_label', 'time', 'frequency', 'miles_approx', 'n_stations_approx']
).properties(width=chart_width, height=chart_height)

highlight1 = alt.Chart(df, title=title).mark_circle(size=200, fillOpacity=0, strokeWidth=3).encode(
    x=alt.X('miles_approx', title='Distance (mi)'),
    y=alt.Y('time', title='Duration (hrs)'),
    stroke=alt.condition(highlight, 'route:N', alt.value('transparent'), legend=None)
).add_params(highlight)

chart_1 = base1 + highlight1

# -----------------------------
# Chart 2: Stations vs Miles
# -----------------------------
base2 = alt.Chart(df, title=title).mark_circle(size=circle_size).encode(
    x=alt.X('miles_approx', title='Distance (mi)'),
    y=alt.Y('n_stations_approx', title='# of Stations'),
    color=alt.Color('route:N', legend=None, scale=alt.Scale(
        domain=list(route_colors.keys()),
        range=list(route_colors.values())
    )),
    tooltip=['route_label', 'time', 'frequency', 'miles_approx', 'n_stations_approx']
).properties(width=chart_width, height=chart_height)

highlight2 = alt.Chart(df, title=title).mark_circle(size=200, fillOpacity=0, strokeWidth=3).encode(
    x=alt.X('miles_approx', title='Distance (mi)'),
    y=alt.Y('n_stations_approx', title='# of Stations'),
    stroke=alt.condition(highlight, 'route:N', alt.value('transparent'), legend=None, )
).add_params(highlight)

chart_2 = base2 + highlight2

# -----------------------------
# Chart 3: Time vs Stations
# -----------------------------
base3 = alt.Chart(df).mark_circle(size=circle_size).encode(
    x=alt.X('n_stations_approx', title="# of Stations"),
    y=alt.Y('time', title='Duration (hrs)'),
    color=alt.Color('route:N', legend=None, scale=alt.Scale(
        domain=list(route_colors.keys()),
        range=list(route_colors.values())
    )),
    tooltip=['route_label', 'time', 'frequency', 'miles_approx', 'n_stations_approx']
).properties(width=chart_width, height=chart_height)

highlight3 = alt.Chart(df, title=title).mark_circle(size=200, fillOpacity=0, strokeWidth=3).encode(
    x='n_stations_approx',
    y='time',
    stroke=alt.condition(highlight, 'route:N', alt.value('transparent'), legend=None)
).add_params(highlight)

chart_3 = base3 + highlight3


final_chart = alt.vconcat(alt.hconcat(chart_1, chart_2, chart_3))
final_chart

alt.VConcatChart(...)

In [82]:
final_chart.save('scatterplots.html')

# Ridership data

In [100]:
df_ridership = pd.read_csv("data/ridership_bts_2005_2025.csv")

df_ridership = df_ridership.rename(
    columns={'State':'state', 'Station': 'station', 
             'Note':'note', 'Value':'value', 
             'Fiscal Year':'fiscal_year'})

df_ridership['station_name']=df_ridership['station'].str.split(',', n=1).str[0]

# adding column names so it can be easily voronoi'd!
df_ridership['group']=df_ridership['state']
df_ridership['subgroup']=df_ridership['station_name']

In [106]:
import bokeh.palettes
bokeh.palettes.Category20[20]

('#1f77b4',
 '#aec7e8',
 '#ff7f0e',
 '#ffbb78',
 '#2ca02c',
 '#98df8a',
 '#d62728',
 '#ff9896',
 '#9467bd',
 '#c5b0d5',
 '#8c564b',
 '#c49c94',
 '#e377c2',
 '#f7b6d2',
 '#7f7f7f',
 '#c7c7c7',
 '#bcbd22',
 '#dbdb8d',
 '#17becf',
 '#9edae5')

In [102]:
years = df_ridership['fiscal_year'].unique()

for YEAR in years:
    df_year = df_ridership.loc[df_ridership['fiscal_year'] == YEAR]
    filepath = f'data/DF_BTS_AMTRAK_RIDERSHIP_{YEAR}.csv'
    df_year.to_csv(filepath, index=False)
    print(f'Saved year {YEAR} to {filepath}.')

Saved year 2005 to data/DF_BTS_AMTRAK_RIDERSHIP_2005.csv.
Saved year 2006 to data/DF_BTS_AMTRAK_RIDERSHIP_2006.csv.
Saved year 2007 to data/DF_BTS_AMTRAK_RIDERSHIP_2007.csv.
Saved year 2008 to data/DF_BTS_AMTRAK_RIDERSHIP_2008.csv.
Saved year 2009 to data/DF_BTS_AMTRAK_RIDERSHIP_2009.csv.
Saved year 2010 to data/DF_BTS_AMTRAK_RIDERSHIP_2010.csv.
Saved year 2011 to data/DF_BTS_AMTRAK_RIDERSHIP_2011.csv.
Saved year 2012 to data/DF_BTS_AMTRAK_RIDERSHIP_2012.csv.
Saved year 2013 to data/DF_BTS_AMTRAK_RIDERSHIP_2013.csv.
Saved year 2014 to data/DF_BTS_AMTRAK_RIDERSHIP_2014.csv.
Saved year 2015 to data/DF_BTS_AMTRAK_RIDERSHIP_2015.csv.
Saved year 2016 to data/DF_BTS_AMTRAK_RIDERSHIP_2016.csv.
Saved year 2017 to data/DF_BTS_AMTRAK_RIDERSHIP_2017.csv.
Saved year 2018 to data/DF_BTS_AMTRAK_RIDERSHIP_2018.csv.
Saved year 2019 to data/DF_BTS_AMTRAK_RIDERSHIP_2019.csv.
Saved year 2020 to data/DF_BTS_AMTRAK_RIDERSHIP_2020.csv.
Saved year 2021 to data/DF_BTS_AMTRAK_RIDERSHIP_2021.csv.
Saved year 202